# Servidor de suma de matrices (PyTorch + GPU) en Colab

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.

Este notebook corre **solo el servidor**. La interfaz PySide6 se queda en tu PC (VS Code) y se conecta por un túnel TCP.

Orden de ejecución (de arriba hacia abajo):
1. Verificar la GPU (y copiar los datos de hardware para el documento técnico)
2. Crear los archivos del servidor
3. **Benchmark** → tabla de tiempos CPU vs CUDA para el documento
4. Abrir el túnel → te da `SERVER_HOST` y `SERVER_PORT` para el `.env` de tu PC
5. Arrancar el servidor (esta celda se queda corriendo y muestra los logs; para detenerlo, botón ■)

⚠️ Mientras la celda del servidor corre no puedes ejecutar otras. Si quieres repetir el benchmark, detén el servidor primero.

## 1. Verificar la GPU

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("\n--- Datos del hardware (para tu documento técnico) ---")
    print("GPU:", p.name)
    print("Capacidad de cómputo (compute capability):", f"{p.major}.{p.minor}")
    print("Multiprocesadores (SM):", p.multi_processor_count)
    print("Memoria de la GPU:", f"{p.total_memory / 1024**3:.1f} GiB")
    print("Versión de CUDA de PyTorch:", torch.version.cuda)
else:
    print("⚠️ No hay GPU. Revisa: Entorno de ejecución → Cambiar tipo de entorno → GPU.")

!nvidia-smi

PyTorch: 2.14.1+cu130
CUDA disponible: True

--- Datos del hardware (para tu documento técnico) ---
GPU: Tesla T4
Capacidad de cómputo (compute capability): 7.5
Multiprocesadores (SM): 40
Memoria de la GPU: 14.6 GiB
Versión de CUDA de PyTorch: 13.0
Wed Nov 25 05:42:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |       

## 2. Crear los archivos del servidor
Son los mismos archivos de tu proyecto (copias exactas).

In [ ]:
%%writefile procesador_matrices.py
"""
procesador_matrices.py  —  CAPA DE PROCESAMIENTO (matemáticas con PyTorch)
--------------------------------------------------------------------------
Clase independiente que suma matrices usando PyTorch en el mejor dispositivo
disponible. NO sabe nada de sockets, JSON ni PySide6: recibe matrices y
devuelve resultados. Por eso se puede reutilizar para otras operaciones
(resta, multiplicación, etc.) sin tocar el servidor.

CONCEPTOS CLAVE
---------------
* Tensor: arreglo n-dimensional de números. Una matriz es un tensor de 2
  dimensiones (filas x columnas). Un escalar es de 0 dim, un vector de 1 dim.
* Dispositivo: dónde "viven" los tensores y dónde se calcula.
      CUDA -> GPU NVIDIA        MPS -> GPU de Apple (Metal)        CPU -> procesador
  Orden de preferencia:  CUDA -> MPS -> CPU
* C = A + B se hace con el operador "+" de PyTorch (operación VECTORIZADA):
  en una GPU, miles de hilos suman elementos distintos a la vez. Nunca usamos
  ciclos for anidados de Python.
"""
import base64
import io
import platform
import threading
import time
from dataclasses import dataclass

import numpy as np
import torch
from PIL import Image


# ----------------------------------------------------------------------
# Selección del dispositivo
# ----------------------------------------------------------------------
def seleccionar_dispositivo(forzar=None):
    """
    Devuelve el torch.device a usar, siguiendo CUDA -> MPS -> CPU.
    `forzar` ("cpu", por ejemplo) sirve para comparar tiempos en el benchmark.
    """
    if forzar is not None:
        return torch.device(forzar)
    if torch.cuda.is_available():            # ¿hay GPU NVIDIA con CUDA?
        return torch.device("cuda")
    if torch.backends.mps.is_available():    # ¿hay GPU Apple con Metal?
        return torch.device("mps")
    return torch.device("cpu")               # alternativa universal


def describir_dispositivo(dispositivo):
    """Nombre legible del hardware (ej. 'Tesla T4')."""
    if dispositivo.type == "cuda":
        return torch.cuda.get_device_name(0)
    if dispositivo.type == "mps":
        return "GPU Apple (Metal)"
    return platform.processor() or platform.machine() or "CPU"


# ----------------------------------------------------------------------
# Resultado de una suma (un contenedor simple de datos)
# ----------------------------------------------------------------------
@dataclass
class ResultadoSuma:
    resultado: np.ndarray     # matriz C ya en CPU (numpy), float32
    filas: int
    columnas: int
    dispositivo: str          # "CUDA", "MPS" o "CPU"
    detalle: str              # nombre del hardware
    tiempo_suma_ms: float     # SOLO la operación C = A + B
    tiempo_total_ms: float    # crear tensores + transferencias + suma + traer resultado


# ----------------------------------------------------------------------
# Clase principal
# ----------------------------------------------------------------------
class ProcesadorMatrices:
    def __init__(self, forzar_dispositivo=None):
        self.dispositivo = seleccionar_dispositivo(forzar_dispositivo)
        self.nombre = self.dispositivo.type.upper()          # "CUDA" / "MPS" / "CPU"
        self.detalle = describir_dispositivo(self.dispositivo)

        # Varios hilos del servidor pueden pedir sumas a la vez. La GPU ejecuta
        # el trabajo en una cola, así que si dos hilos midieran tiempos al
        # mismo tiempo, cada uno mediría también el trabajo del otro. Este
        # candado hace que cada suma se mida "limpia" (una a la vez).
        self._candado = threading.Lock()

        self._calentar()

    # ------------------------------------------------------------------
    def _calentar(self):
        """
        La PRIMERA operación en CUDA es lenta (inicializa el contexto, carga
        kernels). Hacemos una suma diminuta al arrancar para que ese costo no
        contamine la primera medición real.
        """
        a = torch.ones(8, 8, dtype=torch.float32, device=self.dispositivo)
        _ = a + a
        self._sincronizar()

    def _sincronizar(self):
        """
        Las GPU trabajan de forma ASÍNCRONA: Python lanza la operación y sigue
        de largo. Sin synchronize() mediríamos solo el tiempo de "lanzar" la
        suma, no de ejecutarla.
        """
        if self.dispositivo.type == "cuda":
            torch.cuda.synchronize()
        elif self.dispositivo.type == "mps":
            torch.mps.synchronize()

    def liberar_memoria(self):
        """Útil después de un error de memoria (OOM) en la GPU."""
        if self.dispositivo.type == "cuda":
            torch.cuda.empty_cache()
        elif self.dispositivo.type == "mps":
            torch.mps.empty_cache()

    # ------------------------------------------------------------------
    def _a_tensor(self, matriz):
        """Convierte lista de listas / numpy / tensor a un tensor float32 EN el dispositivo."""
        if isinstance(matriz, torch.Tensor):
            tensor = matriz.to(device=self.dispositivo, dtype=torch.float32)
        else:
            # torch.tensor copia los datos y los coloca directo en el dispositivo
            tensor = torch.tensor(matriz, dtype=torch.float32, device=self.dispositivo)
        if tensor.ndim != 2:
            raise ValueError("Cada matriz debe tener exactamente 2 dimensiones")
        return tensor

    def generar_aleatorias(self, filas, columnas):
        """
        Crea dos matrices aleatorias float32 DIRECTAMENTE en el dispositivo
        (evita construir 16 millones de números en Python y copiarlos a la GPU).
        Valores en [0, 1).
        """
        a = torch.rand(filas, columnas, dtype=torch.float32, device=self.dispositivo)
        b = torch.rand(filas, columnas, dtype=torch.float32, device=self.dispositivo)
        return a, b

    # ------------------------------------------------------------------
    def sumar(self, a, b):
        """
        C = A + B, donde C[i][j] = A[i][j] + B[i][j].
        `a` y `b` pueden ser listas, arreglos numpy o tensores.
        """
        with self._candado:
            inicio_total = time.perf_counter()

            # 1) Crear tensores y moverlos al dispositivo
            ta = self._a_tensor(a)
            tb = self._a_tensor(b)
            if ta.shape != tb.shape:
                raise ValueError(f"Las matrices deben tener las mismas dimensiones: {tuple(ta.shape)} vs {tuple(tb.shape)}")
            self._sincronizar()

            # 2) La suma en sí (lo único que medimos como "tiempo de suma")
            inicio_suma = time.perf_counter()
            tc = ta + tb
            self._sincronizar()
            fin_suma = time.perf_counter()

            # 3) Traer el resultado a la CPU (numpy) para poder enviarlo por red
            resultado = tc.cpu().numpy()
            fin_total = time.perf_counter()

        return ResultadoSuma(
            resultado=resultado,
            filas=resultado.shape[0],
            columnas=resultado.shape[1],
            dispositivo=self.nombre,
            detalle=self.detalle,
            tiempo_suma_ms=(fin_suma - inicio_suma) * 1000,
            tiempo_total_ms=(fin_total - inicio_total) * 1000,
        )


# ----------------------------------------------------------------------
# Representación visual (responsabilidad aparte: no es matemática)
# ----------------------------------------------------------------------
def matriz_a_png_base64(matriz, lado_maximo=512):
    """
    Convierte una matriz en una imagen PNG en escala de grises.

    ¿Por qué normalizar? Un píxel solo admite enteros de 0 a 255, pero la
    matriz puede tener cualquier rango (por ejemplo 0.3 a 1.9, o -500 a 900).
    Min-max lleva el mínimo a 0 (negro) y el máximo a 255 (blanco):
            pixel = (valor - mínimo) / (máximo - mínimo) * 255

    Si la matriz es muy grande (ej. 4096x4096) se reduce promediando bloques
    (resampling BOX) para que la imagen viaje rápido por la red.

    Devuelve (png_en_base64, mínimo, máximo, ancho_img, alto_img).
    """
    m = np.asarray(matriz, dtype=np.float32)
    minimo, maximo = float(m.min()), float(m.max())
    rango = maximo - minimo

    if rango == 0:                      # matriz constante: evitamos dividir entre 0
        normalizada = np.zeros_like(m)
    else:
        normalizada = (m - minimo) / rango

    pixeles = (normalizada * 255).astype(np.uint8)
    imagen = Image.fromarray(pixeles)   # uint8 2D -> escala de grises ("L")

    if max(imagen.size) > lado_maximo:
        imagen.thumbnail((lado_maximo, lado_maximo), Image.Resampling.BOX)

    buffer = io.BytesIO()
    imagen.save(buffer, format="PNG")
    png_b64 = base64.b64encode(buffer.getvalue()).decode("ascii")
    return png_b64, minimo, maximo, imagen.size[0], imagen.size[1]


Overwriting procesador_matrices.py


In [ ]:
%%writefile protocolo.py
"""
protocolo.py
------------
Protocolo de comunicación compartido por el cliente y el servidor.

Problema: TCP es un flujo de bytes, no de "mensajes". Una respuesta grande
puede llegar partida en varios recv(). Solución clásica: ANTES de cada mensaje
se envían 4 bytes que indican cuántos bytes mide el mensaje (prefijo de longitud).

Formato de cada mensaje:
    [4 bytes: longitud (big-endian)] [N bytes: JSON en UTF-8]

NOTA: si tu servidor de actividades anteriores ya tiene su propio protocolo,
usa ese en lugar de este archivo. Lo importante es que cliente y servidor
usen el mismo.
"""
import json
import struct


def enviar_json(sock, objeto):
    """Convierte un diccionario a JSON y lo envía con su prefijo de longitud."""
    datos = json.dumps(objeto).encode("utf-8")
    sock.sendall(struct.pack(">I", len(datos)) + datos)


def _recibir_exacto(sock, cantidad):
    """Lee EXACTAMENTE `cantidad` bytes (repite recv hasta completar)."""
    buffer = b""
    while len(buffer) < cantidad:
        trozo = sock.recv(cantidad - len(buffer))
        if not trozo:  # el otro extremo cerró la conexión
            raise ConnectionError("Conexión cerrada antes de recibir todo el mensaje")
        buffer += trozo
    return buffer


def recibir_json(sock):
    """Lee un mensaje completo y lo devuelve como diccionario de Python."""
    cabecera = _recibir_exacto(sock, 4)
    (longitud,) = struct.unpack(">I", cabecera)
    return json.loads(_recibir_exacto(sock, longitud).decode("utf-8"))


Overwriting protocolo.py


In [ ]:
%%writefile config.py
"""
config.py
---------
Lee la configuración (host, puerto y API Key) desde un archivo externo ".env".

¿Por qué así?
La actividad pide que la API Key NUNCA esté escrita en el código fuente.
Con python-dotenv, las variables del archivo .env se cargan como variables de
entorno y las leemos con os.getenv(). El .env no se entrega con el proyecto.
"""
import os
from pathlib import Path

from dotenv import load_dotenv

# Buscamos el .env junto a este archivo (sin importar desde dónde se ejecute).
load_dotenv(Path(__file__).with_name(".env"))

SERVER_HOST = os.getenv("SERVER_HOST", "127.0.0.1")
SERVER_PORT = int(os.getenv("SERVER_PORT", "5000"))
TMDB_API_KEY = os.getenv("TMDB_API_KEY", "")
# Modo del servidor: "multihilo" (por defecto) o "bloqueante" (solo para comparar)
MODO_SERVIDOR = os.getenv("MODO_SERVIDOR", "multihilo").lower()
# Segundos de espera artificial por solicitud (0 = sin retardo). Solo para pruebas.
RETARDO_SIMULADO = float(os.getenv("RETARDO_SIMULADO", "0"))

# ----------------------------------------------------------------------
# Suma de matrices (valores compartidos por cliente y servidor)
# ----------------------------------------------------------------------
# Dimensión máxima por lado. Si tu GPU/RAM no aguanta 4096, bájala aquí.
MAX_DIM_MATRIZ = 4096
# Hasta este tamaño el usuario puede capturar los valores a mano.
MAX_DIM_MANUAL = 10
# Hasta este tamaño el resultado se devuelve numérico; arriba, como imagen.
MAX_DIM_NUMERICO = 20
# Hilos del ThreadPoolExecutor del servidor.
MAX_WORKERS = int(os.getenv("MAX_WORKERS", "10"))


Overwriting config.py


In [ ]:
%%writefile tmdb_api.py
"""
tmdb_api.py  —  CAPA DE DATOS / INTEGRACIÓN
-------------------------------------------
Única clase que habla con la API externa (The Movie Database, TMDB).
Ninguna otra parte del proyecto hace peticiones HTTP a TMDB.

Endpoints que usamos (documentación: https://developer.themoviedb.org/reference):
    GET /movie/popular      -> películas populares (20 por página)
    GET /genre/movie/list   -> catálogo {id: nombre} de géneros

Cada película en /movie/popular trae (entre otros campos):
    title, release_date ("2024-05-31"), vote_average (0-10),
    vote_count, popularity, genre_ids ([28, 12, ...])

Autenticación: API Key gratuita (v3) enviada como parámetro "api_key".
"""
import requests


class TMDBApi:
    BASE_URL = "https://api.themoviedb.org/3"

    def __init__(self, api_key, idioma="es-MX"):
        if not api_key:
            raise ValueError("Falta TMDB_API_KEY. Revisa tu archivo .env")
        self.api_key = api_key
        self.idioma = idioma

    # ------------------------------------------------------------------
    # Método privado genérico: todas las peticiones pasan por aquí.
    # ------------------------------------------------------------------
    def _get(self, endpoint, params=None):
        parametros = {"api_key": self.api_key, "language": self.idioma}
        if params:
            parametros.update(params)

        # timeout evita que el programa se quede colgado si la API no responde
        respuesta = requests.get(f"{self.BASE_URL}{endpoint}", params=parametros, timeout=10)
        respuesta.raise_for_status()  # lanza excepción si el código HTTP es 4xx/5xx
        return respuesta.json()

    # ------------------------------------------------------------------
    # Métodos públicos
    # ------------------------------------------------------------------
    def obtener_generos(self):
        """Devuelve un diccionario {id_genero: nombre_genero}."""
        datos = self._get("/genre/movie/list")
        return {g["id"]: g["name"] for g in datos["genres"]}

    def obtener_peliculas_populares(self, paginas=3):
        """
        Consulta varias páginas de películas populares y devuelve una lista
        de diccionarios SIMPLES con solo lo que el servidor necesita:
            {titulo, anio, calificacion, votos, popularidad, generos: [nombres]}
        """
        generos = self.obtener_generos()
        peliculas = []

        for pagina in range(1, paginas + 1):
            datos = self._get("/movie/popular", {"page": pagina})
            for p in datos["results"]:
                fecha = p.get("release_date") or ""  # a veces viene vacía
                peliculas.append({
                    "titulo": p["title"],
                    "anio": int(fecha[:4]) if fecha[:4].isdigit() else None,
                    "calificacion": p["vote_average"],
                    "votos": p["vote_count"],
                    "popularidad": p["popularity"],
                    # convertimos ids -> nombres (ej. 28 -> "Acción")
                    "generos": [generos[i] for i in p["genre_ids"] if i in generos],
                })
        return peliculas


Overwriting tmdb_api.py


In [ ]:
%%writefile serverd.py
"""
servidor.py  —  CAPA DE LÓGICA / COMUNICACIÓN (lado servidor)
-------------------------------------------------------------
Recibe solicitudes JSON de los clientes, decide qué hacer según el campo
"tipo" y responde.

ACTIVIDAD: SUMA DE MATRICES CON PYTORCH (CUDA / MPS / CPU)
----------------------------------------------------------
Se añadió el tipo de solicitud "SUMA_MATRICES". La matemática vive en
procesador_matrices.py; este archivo solo valida, delega y responde.
La concurrencia ahora la administra un ThreadPoolExecutor.

Este servidor puede funcionar en dos modos (variable MODO_SERVIDOR del .env):

  * "multihilo"  (por defecto): el hilo principal SOLO acepta conexiones y
                 envía cada cliente a un POOL de hilos (ThreadPoolExecutor).
  * "bloqueante": el hilo principal acepta Y atiende al cliente. Mientras lo
                 atiende NO puede volver a accept(), así que los demás
                 clientes esperan en cola. Existe solo para COMPARAR
                 el comportamiento en el documento técnico.

Formato de solicitud : {"tipo": "peliculas_populares", "paginas": 3}
                       {"tipo": "SUMA_MATRICES", "modo": "valores"|"generar",
                        "filas": 3, "columnas": 3,
                        "matriz_a": [[...]], "matriz_b": [[...]]}   (solo modo "valores")
Formato de respuesta : {"estado": "ok", "datos": {...}}
                    o  {"estado": "error", "mensaje": "..."}
"""
import math
import socket
import threading
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import requests

from config import (MAX_DIM_MANUAL, MAX_DIM_MATRIZ, MAX_DIM_NUMERICO, MAX_WORKERS,
                    MODO_SERVIDOR, RETARDO_SIMULADO, SERVER_HOST, SERVER_PORT, TMDB_API_KEY)
from procesador_matrices import ProcesadorMatrices, matriz_a_png_base64
from protocolo import enviar_json, recibir_json
from tmdb_api import TMDBApi

# Una sola instancia compartida por todos los hilos.
# Es segura de compartir: TMDBApi solo guarda api_key e idioma (nunca se
# modifican) y cada petición usa requests.get(), que no comparte estado.
# Si no hay API key (por ejemplo en Colab, donde solo probamos matrices),
# el servidor arranca igual y solo rechaza las solicitudes de películas.
try:
    api = TMDBApi(TMDB_API_KEY)
except ValueError:
    api = None

# Una sola instancia del procesador de matrices: elige el dispositivo
# (CUDA -> MPS -> CPU) UNA vez al arrancar y la usan todos los hilos.
# Tiene su propio candado interno para medir tiempos sin interferencias.
procesador = ProcesadorMatrices()

# ----------------------------------------------------------------------
# UTILIDADES PARA OBSERVAR LA CONCURRENCIA
# ----------------------------------------------------------------------
# print() desde varios hilos a la vez puede mezclar los mensajes en una
# misma línea. Un Lock garantiza que cada mensaje se imprima completo.
_candado_consola = threading.Lock()


def log(mensaje):
    """Imprime con hora y nombre del hilo que lo emite."""
    hora = time.strftime("%H:%M:%S")
    nombre_hilo = threading.current_thread().name
    with _candado_consola:
        print(f"[{hora}] [{nombre_hilo:<12}] {mensaje}", flush=True)


# ----------------------------------------------------------------------
# LÓGICA DE NEGOCIO (sin cambios respecto a la Unidad 1)
# ----------------------------------------------------------------------
def construir_datos_graficas(peliculas):
    """Convierte la lista de películas en tres estructuras listas para graficar."""

    # --- 1) Barras: Top 10 por calificación (solo con suficientes votos,
    #        para que una película con 3 votos de "10" no gane). ---
    confiables = [p for p in peliculas if p["votos"] >= 100]
    top10 = sorted(confiables, key=lambda p: p["calificacion"], reverse=True)[:10]

    # --- 2) Pastel: cuántas películas hay por género. ---
    conteo = Counter(g for p in peliculas for g in p["generos"])
    principales = conteo.most_common(7)
    otros = sum(conteo.values()) - sum(c for _, c in principales)
    generos = [{"genero": g, "cantidad": c} for g, c in principales]
    if otros > 0:
        generos.append({"genero": "Otros", "cantidad": otros})

    # --- 3) Dispersión: popularidad (x) vs calificación (y). ---
    dispersion = [
        {"titulo": p["titulo"], "popularidad": p["popularidad"], "calificacion": p["calificacion"]}
        for p in peliculas if p["votos"] >= 100
    ]

    return {
        "total": len(peliculas),
        "top_calificadas": [{"titulo": p["titulo"], "calificacion": p["calificacion"]} for p in top10],
        "generos": generos,
        "dispersion": dispersion,
    }


# ----------------------------------------------------------------------
# SUMA DE MATRICES: validación + armado de la respuesta
# ----------------------------------------------------------------------
def _dimension_valida(valor, nombre):
    """Entero entre 1 y MAX_DIM_MATRIZ. (bool es subclase de int: lo rechazamos.)"""
    if not isinstance(valor, int) or isinstance(valor, bool):
        raise ValueError(f"'{nombre}' debe ser un número entero")
    if not 1 <= valor <= MAX_DIM_MATRIZ:
        raise ValueError(f"'{nombre}' debe estar entre 1 y {MAX_DIM_MATRIZ}")
    return valor


def _matriz_valida(matriz, filas, columnas, nombre):
    """Comprueba que sea una lista de `filas` listas de `columnas` números finitos."""
    if not isinstance(matriz, list) or len(matriz) != filas:
        raise ValueError(f"La matriz {nombre} debe tener {filas} filas")
    for i, fila in enumerate(matriz):
        if not isinstance(fila, list) or len(fila) != columnas:
            raise ValueError(f"La fila {i + 1} de la matriz {nombre} debe tener {columnas} columnas")
        for valor in fila:
            if isinstance(valor, bool) or not isinstance(valor, (int, float)) or not math.isfinite(valor):
                raise ValueError(f"La matriz {nombre} contiene un valor no numérico o infinito")
    return matriz


def manejar_suma_matrices(solicitud):
    """
    Valida la solicitud, obtiene A y B (del cliente o generadas aquí), suma con
    PyTorch y arma los `datos` de la respuesta. Lanza ValueError si algo es inválido.
    """
    filas = _dimension_valida(solicitud.get("filas"), "filas")
    columnas = _dimension_valida(solicitud.get("columnas"), "columnas")
    modo = solicitud.get("modo")

    if modo == "valores":
        # El cliente manda los valores: solo se permite para matrices pequeñas.
        if filas > MAX_DIM_MANUAL or columnas > MAX_DIM_MANUAL:
            raise ValueError(f"Con valores del cliente el máximo es {MAX_DIM_MANUAL}x{MAX_DIM_MANUAL}")
        a = _matriz_valida(solicitud.get("matriz_a"), filas, columnas, "A")
        b = _matriz_valida(solicitud.get("matriz_b"), filas, columnas, "B")
    elif modo == "generar":
        # El servidor genera las matrices aleatorias directo en el dispositivo.
        a, b = procesador.generar_aleatorias(filas, columnas)
    else:
        raise ValueError("'modo' debe ser 'valores' o 'generar'")

    r = procesador.sumar(a, b)
    log(f"  Suma {filas}x{columnas} en {r.dispositivo} ({r.detalle}): "
        f"{r.tiempo_suma_ms:.3f} ms (total {r.tiempo_total_ms:.1f} ms)")

    datos = {
        "mensaje": "Suma realizada correctamente",
        "filas": filas,
        "columnas": columnas,
        "dispositivo": r.dispositivo,
        "detalle": r.detalle,
        "tiempo_suma_ms": round(r.tiempo_suma_ms, 3),
        "tiempo_total_ms": round(r.tiempo_total_ms, 3),
    }

    if filas <= MAX_DIM_NUMERICO and columnas <= MAX_DIM_NUMERICO:
        # Matriz pequeña: se devuelve como lista de listas.
        datos["formato"] = "numerico"
        datos["resultado"] = r.resultado.tolist()
    else:
        # Matriz grande: se devuelve una imagen en escala de grises.
        png_b64, minimo, maximo, ancho, alto = matriz_a_png_base64(r.resultado)
        datos.update({
            "formato": "imagen",
            "imagen_png_b64": png_b64,
            "imagen_ancho": ancho,
            "imagen_alto": alto,
            "valor_minimo": minimo,
            "valor_maximo": maximo,
        })
    return datos


def manejar_solicitud(solicitud):
    """Despachador: elige qué hacer según el tipo de solicitud."""
    tipo = solicitud.get("tipo")

    if tipo == "peliculas_populares":
        if api is None:
            return {"estado": "error", "mensaje": "El servidor no tiene TMDB_API_KEY configurada"}
        # Limitamos las páginas (1-5) para no abusar de la API.
        paginas = max(1, min(int(solicitud.get("paginas", 3)), 5))
        try:
            peliculas = api.obtener_peliculas_populares(paginas)
            return {"estado": "ok", "datos": construir_datos_graficas(peliculas)}
        except requests.RequestException as error:
            return {"estado": "error", "mensaje": f"Error al consultar TMDB: {error}"}

    if tipo == "SUMA_MATRICES":
        try:
            return {"estado": "ok", "datos": manejar_suma_matrices(solicitud)}
        except ValueError as error:                    # solicitud inválida
            return {"estado": "error", "mensaje": f"Solicitud inválida: {error}"}
        except (RuntimeError, MemoryError) as error:   # incluye "CUDA out of memory"
            procesador.liberar_memoria()
            return {"estado": "error", "mensaje": f"No se pudo procesar la suma: {error}"}

    return {"estado": "error", "mensaje": f"Tipo de solicitud desconocido: {tipo}"}


# ----------------------------------------------------------------------
# ATENCIÓN AL CLIENTE
# ----------------------------------------------------------------------
def atender_cliente(conexion, direccion):
    """
    Atiende a UN cliente. En modo multihilo se ejecuta dentro de un hilo
    del pool (Worker_N); en modo bloqueante, dentro del hilo principal.

    Cada ejecución trabaja con SU conexión y es responsable de cerrarla.
    """
    log(f"Atendiendo al cliente {direccion}")
    try:
        solicitud = recibir_json(conexion)
        log(f"Solicitud recibida: {str(solicitud)[:200]}")  # recortada: puede traer matrices

        # --- Inicio del procesamiento ---
        inicio = time.perf_counter()
        log("▶ INICIA procesamiento")

        # Retardo artificial (solo para pruebas). time.sleep() NO bloquea a
        # los demás hilos: solo "duerme" a este.
        if RETARDO_SIMULADO > 0:
            log(f"  (retardo simulado de {RETARDO_SIMULADO}s)")
            time.sleep(RETARDO_SIMULADO)

        respuesta = manejar_solicitud(solicitud)
        enviar_json(conexion, respuesta)

        # --- Fin del procesamiento ---
        duracion = time.perf_counter() - inicio
        log(f"■ TERMINA procesamiento ({duracion:.2f}s) -> estado: {respuesta['estado']}")

    except Exception as error:  # nunca dejamos que un cliente tumbe el servidor
        log(f"[!] Error con {direccion}: {error}")
    finally:
        conexion.close()  # cada atención cierra SU conexión
        log(f"Conexión cerrada con {direccion}")


# ----------------------------------------------------------------------
# ACEPTACIÓN DE CONEXIONES (hilo principal)
# ----------------------------------------------------------------------
def main():
    # El hilo principal se llama "MainThread" por defecto; lo renombramos
    # para que en los logs quede claro quién acepta y quién atiende.
    threading.current_thread().name = "Principal"

    # ThreadPoolExecutor: crea MAX_WORKERS hilos UNA vez y los reutiliza.
    # Si llegan más clientes que hilos, las tareas esperan en una cola interna
    # (en vez de crear hilos sin límite). Los hilos se llaman Worker_0, Worker_1...
    pool = ThreadPoolExecutor(max_workers=MAX_WORKERS, thread_name_prefix="Worker")

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as servidor:
        servidor.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        servidor.bind((SERVER_HOST, SERVER_PORT))
        servidor.listen()
        log(f"Servidor escuchando en {SERVER_HOST}:{SERVER_PORT} | modo: {MODO_SERVIDOR} | hilos: {MAX_WORKERS}")
        log(f"Dispositivo para matrices: {procesador.nombre} ({procesador.detalle})")

        try:
            while True:
                # accept() bloquea SOLO hasta que llega un cliente nuevo.
                conexion, direccion = servidor.accept()
                log(f"Conexión ACEPTADA de {direccion}")

                if MODO_SERVIDOR == "bloqueante":
                    # --- MODELO ORIGINAL: se atiende aquí mismo. ---
                    # Hasta que atender_cliente() termine, no se vuelve a accept().
                    atender_cliente(conexion, direccion)
                else:
                    # --- MODELO CON POOL: entregamos la tarea y regresamos de inmediato. ---
                    # submit() NO espera a que termine (no es como hilo.join()).
                    pool.submit(atender_cliente, conexion, direccion)
                    log(f"Tarea enviada al pool para {direccion}; vuelvo a escuchar")
        except KeyboardInterrupt:
            log("Servidor detenido por el usuario")
        finally:
            pool.shutdown(wait=False, cancel_futures=True)


if __name__ == "__main__":
    main()


Overwriting serverd.py


In [ ]:
# Dependencias que Colab no trae y configuración (.env) del servidor.
!pip -q install python-dotenv

from getpass import getpass

# La API key de TMDB es OPCIONAL aquí: solo hace falta si quieres probar también la pestaña de películas.
clave = getpass("TMDB_API_KEY (Enter para omitir): ")
with open(".env", "w") as f:
    f.write(f"TMDB_API_KEY={clave}\nSERVER_HOST=127.0.0.1\nSERVER_PORT=5000\nMAX_WORKERS=10\n")
print(".env creado")

TMDB_API_KEY (Enter para omitir): ··········
.env creado


## 3. Benchmark: tiempos CPU vs CUDA (tabla del documento técnico)
Usa la misma clase `ProcesadorMatrices` del servidor. Se toma la **mediana** de varias repeticiones, después de un calentamiento.
- *Suma*: solo `C = A + B`.
- *Total*: incluye además traer el resultado a la CPU (en el servidor real también se suma crear tensores / generar la imagen).

In [ ]:
import statistics
import pandas as pd
import torch
from procesador_matrices import ProcesadorMatrices

TAMANOS = [(3, 3), (20, 20), (256, 256), (1024, 1024), (2048, 2048), (4096, 4096)]
REPETICIONES = 10

procesadores = {"CPU": ProcesadorMatrices(forzar_dispositivo="cpu")}
if torch.cuda.is_available():
    procesadores["CUDA"] = ProcesadorMatrices()

def medir(proc, filas, columnas):
    a, b = proc.generar_aleatorias(filas, columnas)
    proc.sumar(a, b)  # una pasada de calentamiento
    corridas = [proc.sumar(a, b) for _ in range(REPETICIONES)]
    return (statistics.median(r.tiempo_suma_ms for r in corridas),
            statistics.median(r.tiempo_total_ms for r in corridas))

filas_tabla = []
for f, c in TAMANOS:
    fila = {"Dimensiones": f"{f}x{c}", "Elementos": f * c, "Memoria por matriz (MiB)": round(f * c * 4 / 1024**2, 3)}
    for nombre, proc in procesadores.items():
        suma, total = medir(proc, f, c)
        fila[f"{nombre} suma (ms)"] = round(suma, 4)
        fila[f"{nombre} total (ms)"] = round(total, 3)
    if "CUDA" in procesadores:
        fila["Aceleración (suma)"] = round(fila["CPU suma (ms)"] / fila["CUDA suma (ms)"], 1)
    filas_tabla.append(fila)

tabla = pd.DataFrame(filas_tabla)
display(tabla)
print(tabla.to_markdown(index=False))  # versión lista para pegar en el documento

,Dimensiones,Elementos,Memoria por matriz (MiB),CPU suma (ms),CPU total (ms),CUDA suma (ms),CUDA total (ms),Aceleración (suma)
0,3x3,9,0.000,0.0039,0.011,0.0246,0.066,0.2
1,20x20,400,0.002,0.0034,0.010,0.0260,0.072,0.1
2,256x256,65536,0.250,0.0478,0.056,0.0299,0.151,1.6
3,1024x1024,1048576,4.000,1.5072,1.526,0.0744,1.109,20.3
4,2048x2048,4194304,16.000,9.3682,9.394,0.2313,10.929,40.5
5,4096x4096,16777216,64.000,37.8510,37.912,0.8638,51.986,43.8


| Dimensiones   |   Elementos |   Memoria por matriz (MiB) |   CPU suma (ms) |   CPU total (ms) |   CUDA suma (ms) |   CUDA total (ms) |   Aceleración (suma) |
|:--------------|------------:|---------------------------:|----------------:|-----------------:|-----------------:|------------------:|---------------------:|
| 3x3           |           9 |                      0     |          0.0039 |            0.011 |           0.0246 |             0.066 |                  0.2 |
| 20x20         |         400 |                      0.002 |          0.0034 |            0.01  |           0.026  |             0.072 |                  0.1 |
| 256x256       |       65536 |                      0.25  |          0.0478 |            0.056 |           0.0299 |             0.151 |                  1.6 |
| 1024x1024     |     1048576 |                      4     |          1.5072 |            1.526 |           0.0744 |             1.109 |                 20.3 |
| 2048x2048     |     4194304 |         

In [ ]:
import os, subprocess

# Matar cualquier proceso que use el puerto 5000
try:
    resultado = subprocess.run(["fuser", "-k", "5000/tcp"], capture_output=True, text=True)
    print("fuser:", resultado.stdout, resultado.stderr)
except Exception as e:
    print("fuser no disponible, intentando alternativa:", e)

# También matar cualquier python que ejecute serverd.py
subprocess.run(["pkill", "-f", "serverd.py"], capture_output=True)

import time
time.sleep(2)
print("Puerto 5000 liberado. Listo para reiniciar.")

fuser:  10599 5000/tcp:           

Puerto 5000 liberado. Listo para reiniciar.


In [ ]:
import subprocess, time, re, o

# Arrancar servidor en segundo plano
with open("server.log", "w") as f:
    p = subprocess.Popen(["python", "serverd.py"], stdout=f, stderr=subprocess.STDOUT)
with open("server.pid", "w") as f:
    f.write(str(p.pid))
print("Servidor iniciado (PID", p.pid, "). Esperando 3s...")
time.sleep(3)

# Verificar que arrancó
if os.path.exists("server.log"):
    print("--- Logs del servidor ---")
    print(open("server.log").read()[-800:])

# Arrancar pinggy en segundo plano
with open("pinggy.log", "w") as f:
    tunel = subprocess.Popen(
        ["ssh", "-p", "443", "-R0:localhost:5000", "-o", "StrictHostKeyChecking=no", "tcp@a.pinggy.io"],
        stdout=f, stderr=subprocess.STDOUT
    )
print("Pinggy iniciado (PID", tunel.pid, "). Esperando dirección...")

# Esperar a que aparezca tcp://host:puerto
direccion = None
for _ in range(40):
    time.sleep(1)
    if os.path.exists("pinggy.log"):
        contenido = open("pinggy.log").read()
        m = re.search(r"tcp://([\w.\-]+):(\d+)", contenido)
        if m:
            direccion = m.groups()
            break

if direccion:
    host, puerto = direccion
    print("\n" + "="*55)
    print("✅ COPIA ESTO EN EL .env DE TU PC (VS Code):")
    print(f"SERVER_HOST={host}")
    print(f"SERVER_PORT={puerto}")
    print("="*55 + "\n")
    print("⚠️  NO detengas esta celda mientras uses la interfaz.")
else:
    print("❌ No se obtuvo la dirección. Últimas líneas de pinggy.log:")
    print(open("pinggy.log").read()[-1200:])

: 